# Medicare Inpatient (IP) Full Downstream Evaluation

**Owner**: Clinical & Social Determinants Intelligence (CSDI)
**Last Updated**: April 2026

## Objective
Compare three data sources for Medicare inpatient admission prediction (ip6) using the same XGBoost modeling pipeline:

| Model | Description | Features |
|---|---|---|
| **Model 1** | New TE Embedding Only | 256-dim new TE embeddings (round10 exp2b) |
| **Model 2** | New TE + Baseline (Feature Selected) | 48 selected new TE embeddings + 52 baseline features |
| **Model 3** | Production (Baseline + Prod TE) | 48 production TE embeddings + 52 baseline features |

## Data Sources
- **New TE Embeddings**: `edp-prod-storage.edp_ent_sdoheir_cns.a834793_exp_round10_exp2b_medicare_embeddings_20241120_20250930`
- **Production Features + Production TE**: `anbc-hcb-prod.clin_analytics_hcb_prod.inpatient_me_features_history`

## Evaluation Design
- **Training period**: 2024-07-01 to 2025-06-30 (70% train / 15% val / 15% test, stratified random split)
- **Out-of-Time (OOT) test**: 2025-07-01 to 2025-09-30 (3 months)
- **Target**: `ip6` — binary inpatient admission within 6 months

In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import random
import warnings
import joblib
import gc
from io import BytesIO

from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve, confusion_matrix,
    classification_report, brier_score_loss, precision_recall_curve,
    auc, precision_score, average_precision_score
)

import xgboost as xgb
from xgboost import XGBClassifier

from scipy.stats import percentileofscore

from google.cloud import storage
from google.cloud import bigquery
import google.auth

import time
from datetime import datetime

random.seed(123)
np.random.seed(123)
warnings.filterwarnings("ignore")
client = bigquery.Client()

## Configuration

In [ ]:
# === DATE RANGES ===
TRAIN_START = '2024-07-01'
TRAIN_END = '2025-06-30'
OOT_START = '2025-07-01'
OOT_END = '2025-09-30'

# === TABLE NAMES ===
NEW_TE_EMBEDDING_TABLE = 'edp-prod-storage.edp_ent_sdoheir_cns.a834793_exp_round10_exp2b_medicare_embeddings_20241120_20250930'
PRODUCTION_FEATURES_TABLE = 'anbc-hcb-prod.clin_analytics_hcb_prod.inpatient_me_features_history'

# Materialize this table first using:
# data_ingestion/Formal_training_full_downstream/medicare/
# medicare_formal_training_full_downstream_prod_features_outcomes.sql
PRODUCTION_FEATURES_OUTCOMES_TABLE = 'edp-prod-storage.edp_ent_sdoheir_cns.a834793_Medicare_formal_training_full_downstream_prod_features_outcomes_exp_round10_exp2b'

# TODO: Set this to your outcomes table that contains ip6 and mon_6_include columns
# This table should have: individual_id, index_dt, ip6 (binary 0/1), mon_6_include (1=full observation window)
OUTCOMES_TABLE = 'edp-prod-storage.edp_ent_sdoheir_cns.a834793_Medicare_outcome_6mo_final_exp_round10_exp2b'

# === FEATURE DEFINITIONS ===

# All 256 new TE embedding features
NEW_TE_ALL_FEATURES = [f'embedding_{i}' for i in range(256)]

# 48 production-selected embedding dimensions (from new TE table)
SELECTED_NEW_TE_FEATURES = [
    'embedding_6', 'embedding_7', 'embedding_14', 'embedding_20', 'embedding_23',
    'embedding_30', 'embedding_31', 'embedding_36', 'embedding_43', 'embedding_47',
    'embedding_49', 'embedding_57', 'embedding_64', 'embedding_76', 'embedding_81',
    'embedding_89', 'embedding_94', 'embedding_95', 'embedding_98', 'embedding_104',
    'embedding_110', 'embedding_111', 'embedding_124', 'embedding_126', 'embedding_127',
    'embedding_130', 'embedding_131', 'embedding_138', 'embedding_173', 'embedding_174',
    'embedding_177', 'embedding_178', 'embedding_188', 'embedding_192', 'embedding_195',
    'embedding_203', 'embedding_205', 'embedding_207', 'embedding_212', 'embedding_219',
    'embedding_224', 'embedding_229', 'embedding_230', 'embedding_233', 'embedding_238',
    'embedding_244', 'embedding_253', 'embedding_254'
]

# 48 production TE embedding features (from production table)
PRODUCTION_EMB_FEATURES = [
    'emb6', 'emb7', 'emb14', 'emb20', 'emb23',
    'emb30', 'emb31', 'emb36', 'emb43', 'emb47',
    'emb49', 'emb57', 'emb64', 'emb76', 'emb81',
    'emb89', 'emb94', 'emb95', 'emb98', 'emb104',
    'emb110', 'emb111', 'emb124', 'emb126', 'emb127',
    'emb130', 'emb131', 'emb138', 'emb173', 'emb174',
    'emb177', 'emb178', 'emb188', 'emb192', 'emb195',
    'emb203', 'emb205', 'emb207', 'emb212', 'emb219',
    'emb224', 'emb229', 'emb230', 'emb233', 'emb238',
    'emb244', 'emb253', 'emb254'
]

# 52 baseline features (non-embedding clinical features from production table)
BASELINE_FEATURES = [
    # HPD Chronic Conditions (11)
    'camemhpd_aff', 'camemhpd_alc', 'camemhpd_cbd', 'camemhpd_chf',
    'camemhpd_chr_flag', 'camemhpd_cop', 'camemhpd_crf', 'camemhpd_cv_cond',
    'camemhpd_dia', 'camemhpd_hyp', 'camemhpd_ngd',
    # Medical Utilization (4)
    'camemmedutilization_clm_ln_cnt', 'camemmedutilization_er_clm_cnt',
    'camemmedutilization_uniq_dx_cd_cnt', 'camemmedutilization_uniq_rev_cd_cnt',
    # Medical Case (5)
    'camemmedcasedc1_ip_cnt_dc1', 'camemmedcasedc1_ip_days_dc1',
    'camemmedcasedc2_ip_cnt_dc2', 'camemmedcasedc2_ip_days_dc2',
    'camemmedcasedc3_ip_cnt_dc3',
    # Diagnosis (1)
    'camemeipdxdc1_dxc1085_cnt_dc1',
    # Procedures (5)
    'camemeipprcdc1_prc141_cnt_dc1', 'camemeipprcdc1_prc155_cnt_dc1',
    'camemeipprcdc1_prc219_cnt_dc1', 'camemeipprcdc1_prcc1102_cnt_dc1',
    'camemeipprcdc1_prcc1115_cnt_dc1',
    # Revenue Codes (2)
    'camemrevenuedc3_rev730_cnt_dc3', 'camemrevenuedc4_rev430_cnt_dc4',
    # ER/UC (2)
    'camemerucdc1_erclm_cnt_dc1', 'camemerucdc2_erclm_cnt_dc2',
    # RX Class Utilization (2)
    'camemrxclassutilizationdc5_anticonvulsants_misc_flag_dc5',
    'camemrxclassutilizationdc5_loop_diuretics_flag_dc5',
    # RX Group Utilization (5)
    'camemrxgrouputilizationdc1_antidepressants_days_dc1',
    'camemrxgrouputilizationdc1_corticosteroids_days_dc1',
    'camemrxgrouputilizationdc2_anticonvulsants_days_dc2',
    'camemrxgrouputilizationdc2_corticosteroids_days_dc2',
    'camemrxgrouputilizationdc3_anticonvulsants_flag_dc3',
    # Specialty Claims (3)
    'camemspcclmdc1_spcclmwhos_cnt_dc1', 'camemspcclmdc2_spcclmwhos_cnt_dc2',
    'camemspcofcdc3_spcd_cnt_dc3',
    # Care Management (2)
    'camemtgtptpdc2_cm_soe', 'camemtgtptpdc5_cm_soe',
    # Text Notes (2)
    'camemtxtnotesdc5_txt_end_dc5', 'camemtxtnotesdc5_txt_short_dc5',
    # YLM Demographics (4)
    'camemylm_ylm_homeagesourcer', 'camemylm_ylm_orent',
    'camemylm_ylm_tw_hvalsecinv', 'camemylm_ylm_tw_hvyinvtrad',
    # Membership Demographics (2)
    'camemmbrshp_age65_74', 'camemmbrshp_agenbr',
    # Expenditure (2)
    'e_caperetdem21220', 'e_caperetdem444'
]

# Random seeds (matching original pipeline)
RANDOM_SEED = 123
SPLIT_RANDOM_STATE = 199

print(f"Training period: {TRAIN_START} to {TRAIN_END}")
print(f"OOT test period: {OOT_START} to {OOT_END}")
print(f"\nFeature counts:")
print(f"  Model 1 (New TE only):      {len(NEW_TE_ALL_FEATURES)} features")
print(f"  Model 2 (New TE + Baseline): {len(SELECTED_NEW_TE_FEATURES)} + {len(BASELINE_FEATURES)} = {len(SELECTED_NEW_TE_FEATURES) + len(BASELINE_FEATURES)} features")
print(f"  Model 3 (Production):        {len(PRODUCTION_EMB_FEATURES)} + {len(BASELINE_FEATURES)} = {len(PRODUCTION_EMB_FEATURES) + len(BASELINE_FEATURES)} features")

## Helper Functions

In [ ]:
def preprocess_dataframe(df):
    """Preprocess dataframe: handle nulls and type casting.
    Matches the original pipeline preprocessing exactly.
    """
    for col in df.columns:
        if df[col].dtype in ['float64', 'float32']:
            df[col] = df[col].fillna(0.0).astype(float)
        elif df[col].dtype == 'int64':
            df[col] = df[col].fillna(0).astype(int)
        elif df[col].dtype == 'object':
            df[col] = df[col].replace('', 0).fillna(0)
            try:
                df[col] = df[col].astype(float)
            except (ValueError, TypeError):
                print(f"Warning: Could not convert column {col} to numeric, keeping as object")
    return df



def downcast_numeric_frame(df):
    """Reduce DataFrame memory immediately after BigQuery ingestion."""
    float_cols = df.select_dtypes(include=["float64"]).columns
    int_cols = df.select_dtypes(include=["int64"]).columns

    if len(float_cols) > 0:
        df[float_cols] = df[float_cols].astype(np.float32)
    if len(int_cols) > 0:
        for column in int_cols:
            df[column] = pd.to_numeric(df[column], downcast="integer")

    return df



def read_bigquery_table_direct(client, table_name):
    """Read a materialized BigQuery table directly into pandas."""
    table = client.get_table(table_name)
    row_iterator = client.list_rows(table)

    try:
        df = row_iterator.to_dataframe(create_bqstorage_client=True)
    except (ImportError, TypeError, ValueError):
        df = row_iterator.to_dataframe()

    return downcast_numeric_frame(df)

In [ ]:
def calculate_comprehensive_metrics(y_true, y_pred_proba, prefix=''):
    """Calculate comprehensive evaluation metrics matching the original pipeline.
    
    Metrics: AUC-ROC, AUC-PR, Brier Score, Lift/TP/Precision at 1%, 5%, 10%.
    """
    metrics = {}
    
    # Core metrics
    metrics[f'{prefix}auc_roc'] = roc_auc_score(y_true, y_pred_proba)
    metrics[f'{prefix}auc_pr'] = average_precision_score(y_true, y_pred_proba)
    metrics[f'{prefix}brier_score'] = brier_score_loss(y_true, y_pred_proba)
    
    # Population stats
    n_samples = len(y_true)
    n_positives = int(y_true.sum())
    prevalence = y_true.mean()
    
    metrics[f'{prefix}n_samples'] = n_samples
    metrics[f'{prefix}n_positives'] = n_positives
    metrics[f'{prefix}prevalence'] = prevalence
    
    # Lift metrics at various percentiles
    sorted_indices = np.argsort(-y_pred_proba)
    y_sorted = np.array(y_true)[sorted_indices]
    
    for pct in [1, 5, 10]:
        k = max(1, int(n_samples * pct / 100))
        top_k_labels = y_sorted[:k]
        tp_at_k = int(top_k_labels.sum())
        precision_at_k = top_k_labels.mean()
        lift_at_k = precision_at_k / prevalence if prevalence > 0 else 0
        
        metrics[f'{prefix}tp_at_{pct}pct'] = tp_at_k
        metrics[f'{prefix}precision_at_{pct}pct'] = precision_at_k
        metrics[f'{prefix}lift_at_{pct}pct'] = lift_at_k
    
    return metrics


def print_metrics_summary(metrics, model_name):
    """Print a formatted summary of model metrics."""
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    for split_name in ['train_', 'val_', 'test_']:
        split_label = split_name.rstrip('_').upper()
        auc_key = f'{split_name}auc_roc'
        if auc_key not in metrics:
            continue
        print(f"\n  {split_label}:")
        print(f"    AUC-ROC:    {metrics[f'{split_name}auc_roc']:.4f}")
        print(f"    AUC-PR:     {metrics[f'{split_name}auc_pr']:.4f}")
        print(f"    Brier:      {metrics[f'{split_name}brier_score']:.4f}")
        print(f"    Lift@1%:    {metrics[f'{split_name}lift_at_1pct']:.2f}x")
        print(f"    Lift@5%:    {metrics[f'{split_name}lift_at_5pct']:.2f}x")
        print(f"    Lift@10%:   {metrics[f'{split_name}lift_at_10pct']:.2f}x")
        print(f"    Samples:    {metrics[f'{split_name}n_samples']:,}")
        print(f"    Positives:  {metrics[f'{split_name}n_positives']:,}")
        print(f"    Prevalence: {metrics[f'{split_name}prevalence']:.4f}")
    
    # Overfitting check
    if 'train_auc_roc' in metrics and 'val_auc_roc' in metrics:
        gap = metrics['train_auc_roc'] - metrics['val_auc_roc']
        if gap > 0.05:
            print(f"\n  WARNING: Potential overfitting detected! Train-Val AUC gap = {gap:.4f}")
        else:
            print(f"\n  Train-Val AUC gap: {gap:.4f} (OK)")

## Data Loading

In [ ]:
# --- Load New TE Embeddings ---
print("Loading new TE embeddings (round10 exp2b)...")
t0 = time.time()

query_new_te = f"""
SELECT *
FROM `{NEW_TE_EMBEDDING_TABLE}`
WHERE index_dt >= '{TRAIN_START}' AND index_dt <= '{OOT_END}'
"""

df_new_te = client.query(query_new_te).to_dataframe()
print(f"New TE embeddings loaded: {df_new_te.shape} in {time.time()-t0:.1f}s")
print(f"Date range: {df_new_te['index_dt'].min()} to {df_new_te['index_dt'].max()}")
print(f"Unique members: {df_new_te['individual_id'].nunique():,}")
df_new_te.head()

In [ ]:
# --- Load Production Features + Outcomes (materialized in BigQuery) ---
print("Loading materialized production features + outcomes table...")
t0 = time.time()

df_prod = read_bigquery_table_direct(
    client=client,
    table_name=PRODUCTION_FEATURES_OUTCOMES_TABLE,
)

print(f"Production features + outcomes loaded: {df_prod.shape} in {time.time()-t0:.1f}s")
print(f"Date range: {df_prod['index_dt'].min()} to {df_prod['index_dt'].max()}")
print(f"Unique members: {df_prod['individual_id'].nunique():,}")

df_prod.head()

In [ ]:
# --- Outcomes / Target Variable ---
# Outcomes are already included in the materialized production features table.
print("Outcomes are included in the materialized production table.")
print(f"Target distribution:\n{df_prod['ip6'].value_counts()}")
print(f"\nPositive rate: {df_prod['ip6'].mean():.4f}")

In [ ]:
# --- Data Inspection ---
print("=== New TE Embedding Table ===")
print(f"Shape: {df_new_te.shape}")
print(f"Columns: {list(df_new_te.columns[:5])} ... {list(df_new_te.columns[-3:])}")
print(f"\n=== Production Features + Outcomes Table ===")
print(f"Shape: {df_prod.shape}")
print(f"Columns ({len(df_prod.columns)}): {list(df_prod.columns)}")

# Verify expected columns exist
missing_baseline = [f for f in BASELINE_FEATURES if f not in df_prod.columns]
missing_prod_emb = [f for f in PRODUCTION_EMB_FEATURES if f not in df_prod.columns]
missing_new_te = [f for f in NEW_TE_ALL_FEATURES if f not in df_new_te.columns]
missing_outcomes = [f for f in ['ip6', 'mon_6_include'] if f not in df_prod.columns]

if missing_baseline:
    print(f"\nWARNING: Missing baseline features in production table: {missing_baseline}")
if missing_prod_emb:
    print(f"WARNING: Missing production embedding features: {missing_prod_emb}")
if missing_new_te:
    print(f"WARNING: Missing new TE embedding features: {missing_new_te}")
if missing_outcomes:
    print(f"WARNING: Missing outcome columns in production table: {missing_outcomes}")
if not missing_baseline and not missing_prod_emb and not missing_new_te and not missing_outcomes:
    print("\nAll expected feature and outcome columns found.")

## Data Preprocessing & Merging

- Inner join on `individual_id` + `index_dt` to ensure same population across all models
- Preprocess: fill NaN with 0, cast types
- Three feature sets defined from the merged data

In [ ]:
# --- Normalize join keys ---
df_new_te['individual_id'] = df_new_te['individual_id'].astype(str)
df_new_te['index_dt'] = pd.to_datetime(df_new_te['index_dt']).dt.normalize()

df_prod['individual_id'] = df_prod['individual_id'].astype(str)
df_prod['index_dt'] = pd.to_datetime(df_prod['index_dt']).dt.normalize()

# --- Merge: new TE embeddings + production features/outcomes ---
print(f"Before merge - New TE: {len(df_new_te):,}, Production + Outcomes: {len(df_prod):,}")

df_merged = pd.merge(
    df_new_te,
    df_prod,
    on=['individual_id', 'index_dt'],
    how='inner',
    suffixes=('', '_prod')
)
print(f"After merge: {len(df_merged):,}")

# Clean up memory
del df_new_te, df_prod
gc.collect()

# --- Preprocess ---
# Identify feature columns to preprocess (exclude keys and target)
all_feature_cols = NEW_TE_ALL_FEATURES + BASELINE_FEATURES + PRODUCTION_EMB_FEATURES
available_feature_cols = [c for c in all_feature_cols if c in df_merged.columns]

# Apply preprocessing to feature columns
df_features = df_merged[available_feature_cols].copy()
df_features = preprocess_dataframe(df_features)
df_merged[available_feature_cols] = df_features
del df_features

# Cast target
df_merged['ip6'] = df_merged['ip6'].astype(int)

print(f"\nFinal merged dataset: {df_merged.shape}")
print(f"Target distribution:")
print(df_merged['ip6'].value_counts())
print(f"Positive rate: {df_merged['ip6'].mean():.4f}")

## Feature Set Definition

In [ ]:
# --- Define three feature sets ---

# Model 1: New TE Embedding Only (256 features)
features_model1 = [f for f in NEW_TE_ALL_FEATURES if f in df_merged.columns]

# Model 2: Selected New TE Embeddings + Baseline Features (48 + 52 = 100 features)
features_model2 = (
    [f for f in SELECTED_NEW_TE_FEATURES if f in df_merged.columns] +
    [f for f in BASELINE_FEATURES if f in df_merged.columns]
)

# Model 3: Production TE Embeddings + Baseline Features (48 + 52 = 100 features)
features_model3 = (
    [f for f in PRODUCTION_EMB_FEATURES if f in df_merged.columns] +
    [f for f in BASELINE_FEATURES if f in df_merged.columns]
)

FEATURE_SETS = {
    'model_1_new_te_only': features_model1,
    'model_2_new_te_hybrid': features_model2,
    'model_3_production': features_model3
}

print("Feature Set Summary:")
print(f"  Model 1 (New TE Only):          {len(features_model1)} features")
print(f"  Model 2 (New TE + Baseline):     {len(features_model2)} features")
print(f"  Model 3 (Production):            {len(features_model3)} features")

# Verify no duplicates
for name, feats in FEATURE_SETS.items():
    if len(feats) != len(set(feats)):
        print(f"WARNING: Duplicate features in {name}!")

## Train / Validation / Test + OOT Split

- **In-time** (2024-07-01 to 2025-06-30): 70% train / 15% validation / 15% test (stratified random)
- **Out-of-time** (2025-07-01 to 2025-09-30): Separate temporal test set

In [ ]:
# --- Temporal split: in-time vs OOT ---
df_merged['index_dt'] = pd.to_datetime(df_merged['index_dt'])

mask_intime = (df_merged['index_dt'] >= TRAIN_START) & (df_merged['index_dt'] <= TRAIN_END)
mask_oot = (df_merged['index_dt'] >= OOT_START) & (df_merged['index_dt'] <= OOT_END)

df_intime = df_merged[mask_intime].copy()
df_oot = df_merged[mask_oot].copy()

print(f"In-time samples:  {len(df_intime):,} ({df_intime['ip6'].mean():.4f} positive rate)")
print(f"OOT samples:      {len(df_oot):,} ({df_oot['ip6'].mean():.4f} positive rate)")

# --- Stratified random split of in-time data ---
# First split: 70% train, 30% temp
y_intime = df_intime['ip6']

train_idx, temp_idx = train_test_split(
    df_intime.index, test_size=0.30, stratify=y_intime, random_state=SPLIT_RANDOM_STATE
)

# Second split: 50/50 of temp -> 15% val, 15% test
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=y_intime.loc[temp_idx], random_state=SPLIT_RANDOM_STATE
)

df_train = df_intime.loc[train_idx]
df_val = df_intime.loc[val_idx]
df_test = df_intime.loc[test_idx]

print(f"\nIn-time split:")
print(f"  Train: {len(df_train):,} ({df_train['ip6'].mean():.4f} positive rate)")
print(f"  Val:   {len(df_val):,} ({df_val['ip6'].mean():.4f} positive rate)")
print(f"  Test:  {len(df_test):,} ({df_test['ip6'].mean():.4f} positive rate)")
print(f"  OOT:   {len(df_oot):,} ({df_oot['ip6'].mean():.4f} positive rate)")

## Model Training

XGBoost with two hyperparameter configurations:
- **Embeddings-only** (Model 1): slightly relaxed regularization, `scale_pos_weight`
- **Baseline / Hybrid** (Models 2 & 3): heavy regularization matching production config

In [ ]:
# --- Hyperparameters (matching original pipeline) ---

# Embeddings-only hyperparameters (for Model 1)
params_embeddings_only = {
    'objective': 'binary:logistic',
    'max_depth': 4,
    'min_child_weight': 50,
    'eta': 0.01,
    'subsample': 0.7,
    'colsample_bytree': 0.5,
    'lambda': 3.0,
    'alpha': 1.0,
    'gamma': 1.0,
    'scale_pos_weight': 14.4,
    'tree_method': 'hist',
    'grow_policy': 'depthwise',
    'eval_metric': 'auc',
    'seed': RANDOM_SEED
}

# Baseline / Hybrid hyperparameters (for Models 2 & 3)
params_hybrid = {
    'objective': 'binary:logistic',
    'max_depth': 3,
    'min_child_weight': 100,
    'eta': 0.005,
    'subsample': 0.6,
    'colsample_bytree': 0.4,
    'lambda': 5.0,
    'alpha': 2.0,
    'gamma': 2.0,
    'max_delta_step': 1,
    'tree_method': 'hist',
    'grow_policy': 'depthwise',
    'eval_metric': 'auc',
    'seed': RANDOM_SEED
}

NUM_BOOST_ROUND = 5000
EARLY_STOPPING_ROUNDS = 100
VERBOSE_EVAL = 100

print("Embeddings-only params:", params_embeddings_only)
print("\nHybrid/Production params:", params_hybrid)

In [ ]:
def train_and_evaluate_model(model_name, features, params, use_scaler=False):
    """Train XGBoost model and return model, predictions, and metrics.
    
    Matches the original pipeline: xgb.train with DMatrix, early stopping on validation AUC.
    """
    print(f"\n{'='*70}")
    print(f"  Training: {model_name}")
    print(f"  Features: {len(features)}")
    print(f"  Scaler:   {use_scaler}")
    print(f"{'='*70}")
    
    # Extract feature matrices
    X_train = df_train[features].values.astype(np.float32)
    X_val = df_val[features].values.astype(np.float32)
    X_test = df_test[features].values.astype(np.float32)
    X_oot = df_oot[features].values.astype(np.float32)
    
    y_train = df_train['ip6'].values
    y_val = df_val['ip6'].values
    y_test = df_test['ip6'].values
    y_oot = df_oot['ip6'].values
    
    # Optional standardization (embeddings-only mode)
    scaler = None
    if use_scaler:
        scaler = preprocessing.StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)
        X_oot = scaler.transform(X_oot)
        print("  StandardScaler applied.")
    
    # Create DMatrix objects
    dtrain = xgb.DMatrix(X_train, y_train, feature_names=features)
    dval = xgb.DMatrix(X_val, y_val, feature_names=features)
    dtest = xgb.DMatrix(X_test, y_test, feature_names=features)
    doot = xgb.DMatrix(X_oot, y_oot, feature_names=features)
    
    # Train
    t0 = time.time()
    bst = xgb.train(
        params,
        dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dtrain, 'train'), (dval, 'val')],
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose_eval=VERBOSE_EVAL
    )
    train_time = time.time() - t0
    print(f"\n  Training completed in {train_time:.1f}s")
    print(f"  Best iteration: {bst.best_iteration}")
    print(f"  Best validation AUC: {bst.best_score:.4f}")
    
    # Predict
    pred_train = bst.predict(dtrain)
    pred_val = bst.predict(dval)
    pred_test = bst.predict(dtest)
    pred_oot = bst.predict(doot)
    
    # Calculate metrics
    all_metrics = {}
    all_metrics.update(calculate_comprehensive_metrics(y_train, pred_train, prefix='train_'))
    all_metrics.update(calculate_comprehensive_metrics(y_val, pred_val, prefix='val_'))
    all_metrics.update(calculate_comprehensive_metrics(y_test, pred_test, prefix='test_'))
    all_metrics.update(calculate_comprehensive_metrics(y_oot, pred_oot, prefix='oot_'))
    all_metrics['best_iteration'] = bst.best_iteration
    all_metrics['train_time_seconds'] = train_time
    
    # Print summary
    print_metrics_summary(all_metrics, model_name)
    
    # OOT metrics
    print(f"\n  OOT:")
    print(f"    AUC-ROC:    {all_metrics['oot_auc_roc']:.4f}")
    print(f"    AUC-PR:     {all_metrics['oot_auc_pr']:.4f}")
    print(f"    Brier:      {all_metrics['oot_brier_score']:.4f}")
    print(f"    Lift@1%:    {all_metrics['oot_lift_at_1pct']:.2f}x")
    print(f"    Lift@5%:    {all_metrics['oot_lift_at_5pct']:.2f}x")
    print(f"    Lift@10%:   {all_metrics['oot_lift_at_10pct']:.2f}x")
    
    return {
        'model': bst,
        'scaler': scaler,
        'features': features,
        'params': params,
        'metrics': all_metrics,
        'predictions': {
            'train': (y_train, pred_train),
            'val': (y_val, pred_val),
            'test': (y_test, pred_test),
            'oot': (y_oot, pred_oot)
        }
    }

In [ ]:
# --- Train Model 1: New TE Embedding Only (256 features) ---
results_model1 = train_and_evaluate_model(
    model_name='Model 1: New TE Embedding Only',
    features=features_model1,
    params=params_embeddings_only,
    use_scaler=True
)

In [ ]:
# --- Train Model 2: New TE (Selected) + Baseline Features (100 features) ---
results_model2 = train_and_evaluate_model(
    model_name='Model 2: New TE + Baseline (Feature Selected)',
    features=features_model2,
    params=params_hybrid,
    use_scaler=False
)

In [ ]:
# --- Train Model 3: Production (Baseline + Production TE) (100 features) ---
results_model3 = train_and_evaluate_model(
    model_name='Model 3: Production (Baseline + Prod TE)',
    features=features_model3,
    params=params_hybrid,
    use_scaler=False
)

## Model Comparison — In-Time

In [ ]:
# --- In-Time Comparison Summary ---
all_results = {
    'Model 1: New TE Only': results_model1,
    'Model 2: New TE + Baseline': results_model2,
    'Model 3: Production': results_model3
}

comparison_rows = []
for model_name, result in all_results.items():
    m = result['metrics']
    row = {
        'Model': model_name,
        'N Features': len(result['features']),
        'Best Iter': m['best_iteration'],
        # Validation metrics
        'Val AUC-ROC': f"{m['val_auc_roc']:.4f}",
        'Val AUC-PR': f"{m['val_auc_pr']:.4f}",
        'Val Brier': f"{m['val_brier_score']:.4f}",
        'Val Lift@1%': f"{m['val_lift_at_1pct']:.2f}x",
        'Val Lift@5%': f"{m['val_lift_at_5pct']:.2f}x",
        'Val Lift@10%': f"{m['val_lift_at_10pct']:.2f}x",
        # Test metrics
        'Test AUC-ROC': f"{m['test_auc_roc']:.4f}",
        'Test AUC-PR': f"{m['test_auc_pr']:.4f}",
        'Test Brier': f"{m['test_brier_score']:.4f}",
        'Test Lift@1%': f"{m['test_lift_at_1pct']:.2f}x",
        'Test Lift@5%': f"{m['test_lift_at_5pct']:.2f}x",
        'Test Lift@10%': f"{m['test_lift_at_10pct']:.2f}x",
        # Overfitting gap
        'Train-Val AUC Gap': f"{m['train_auc_roc'] - m['val_auc_roc']:.4f}"
    }
    comparison_rows.append(row)

df_comparison = pd.DataFrame(comparison_rows)
print("\n=== IN-TIME MODEL COMPARISON ===")
display(df_comparison.set_index('Model').T)

In [ ]:
# --- In-Time Visualizations: ROC & PR Curves ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

colors = {'Model 1: New TE Only': '#2196F3', 
          'Model 2: New TE + Baseline': '#4CAF50', 
          'Model 3: Production': '#FF5722'}

# --- ROC Curves (Test Set) ---
ax = axes[0]
for model_name, result in all_results.items():
    y_true, y_pred = result['predictions']['test']
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    auc_val = roc_auc_score(y_true, y_pred)
    ax.plot(fpr, tpr, label=f"{model_name} (AUC={auc_val:.4f})", color=colors[model_name], linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — In-Time Test Set')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- PR Curves (Test Set) ---
ax = axes[1]
for model_name, result in all_results.items():
    y_true, y_pred = result['predictions']['test']
    precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)
    ax.plot(recall_vals, precision_vals, label=f"{model_name} (AP={ap:.4f})", color=colors[model_name], linewidth=2)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — In-Time Test Set')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Lift Comparison (Test Set) ---
ax = axes[2]
percentiles = [1, 5, 10]
x_pos = np.arange(len(percentiles))
width = 0.25
for i, (model_name, result) in enumerate(all_results.items()):
    m = result['metrics']
    lifts = [m[f'test_lift_at_{p}pct'] for p in percentiles]
    ax.bar(x_pos + i * width, lifts, width, label=model_name, color=colors[model_name])
ax.set_xticks(x_pos + width)
ax.set_xticklabels([f'{p}%' for p in percentiles])
ax.set_ylabel('Lift')
ax.set_title('Lift @ Top Percentiles — In-Time Test Set')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('models/intime_comparison_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Model Comparison — Out-of-Time (OOT)

In [ ]:
# --- OOT Comparison Summary ---
oot_rows = []
for model_name, result in all_results.items():
    m = result['metrics']
    row = {
        'Model': model_name,
        'N Features': len(result['features']),
        'OOT AUC-ROC': f"{m['oot_auc_roc']:.4f}",
        'OOT AUC-PR': f"{m['oot_auc_pr']:.4f}",
        'OOT Brier': f"{m['oot_brier_score']:.4f}",
        'OOT Lift@1%': f"{m['oot_lift_at_1pct']:.2f}x",
        'OOT Lift@5%': f"{m['oot_lift_at_5pct']:.2f}x",
        'OOT Lift@10%': f"{m['oot_lift_at_10pct']:.2f}x",
        'OOT Samples': f"{m['oot_n_samples']:,}",
        'OOT Positives': f"{m['oot_n_positives']:,}",
        'OOT Prevalence': f"{m['oot_prevalence']:.4f}",
        # Temporal stability: test vs OOT AUC gap
        'Test-OOT AUC Gap': f"{m['test_auc_roc'] - m['oot_auc_roc']:.4f}"
    }
    oot_rows.append(row)

df_oot_comparison = pd.DataFrame(oot_rows)
print("\n=== OUT-OF-TIME (OOT) MODEL COMPARISON ===")
display(df_oot_comparison.set_index('Model').T)

In [ ]:
# --- OOT Visualizations: ROC & PR Curves ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- ROC Curves (OOT) ---
ax = axes[0]
for model_name, result in all_results.items():
    y_true, y_pred = result['predictions']['oot']
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    auc_val = roc_auc_score(y_true, y_pred)
    ax.plot(fpr, tpr, label=f"{model_name} (AUC={auc_val:.4f})", color=colors[model_name], linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — OOT Test Set')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- PR Curves (OOT) ---
ax = axes[1]
for model_name, result in all_results.items():
    y_true, y_pred = result['predictions']['oot']
    precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)
    ax.plot(recall_vals, precision_vals, label=f"{model_name} (AP={ap:.4f})", color=colors[model_name], linewidth=2)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — OOT Test Set')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Lift Comparison (OOT) ---
ax = axes[2]
for i, (model_name, result) in enumerate(all_results.items()):
    m = result['metrics']
    lifts = [m[f'oot_lift_at_{p}pct'] for p in percentiles]
    ax.bar(x_pos + i * width, lifts, width, label=model_name, color=colors[model_name])
ax.set_xticks(x_pos + width)
ax.set_xticklabels([f'{p}%' for p in percentiles])
ax.set_ylabel('Lift')
ax.set_title('Lift @ Top Percentiles — OOT Test Set')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('models/oot_comparison_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Combined Comparison: In-Time Test vs OOT ---
print("\n=== COMBINED COMPARISON: IN-TIME TEST vs OOT ===")
combined_rows = []
for model_name, result in all_results.items():
    m = result['metrics']
    row = {
        'Model': model_name,
        'Test AUC-ROC': f"{m['test_auc_roc']:.4f}",
        'OOT AUC-ROC': f"{m['oot_auc_roc']:.4f}",
        'AUC Degradation': f"{m['test_auc_roc'] - m['oot_auc_roc']:.4f}",
        'Test Lift@1%': f"{m['test_lift_at_1pct']:.2f}x",
        'OOT Lift@1%': f"{m['oot_lift_at_1pct']:.2f}x",
        'Test Lift@5%': f"{m['test_lift_at_5pct']:.2f}x",
        'OOT Lift@5%': f"{m['oot_lift_at_5pct']:.2f}x",
        'Test AUC-PR': f"{m['test_auc_pr']:.4f}",
        'OOT AUC-PR': f"{m['oot_auc_pr']:.4f}",
    }
    combined_rows.append(row)

df_combined = pd.DataFrame(combined_rows)
display(df_combined.set_index('Model').T)

## Feature Importance

In [ ]:
# --- Feature Importance Analysis for All Models ---
fig, axes = plt.subplots(1, 3, figsize=(24, 10))

for ax_idx, (model_name, result) in enumerate(all_results.items()):
    bst = result['model']
    importance = bst.get_score(importance_type='gain')
    
    if not importance:
        print(f"No feature importance available for {model_name}")
        continue
    
    # Sort and get top 20
    imp_df = pd.DataFrame([
        {'feature': k, 'importance': v} for k, v in importance.items()
    ]).sort_values('importance', ascending=False)
    
    top_20 = imp_df.head(20)
    
    # Categorize features
    top_20['type'] = top_20['feature'].apply(
        lambda x: 'New TE Embedding' if x.startswith('embedding_') 
        else ('Prod TE Embedding' if x.startswith('emb') else 'Baseline')
    )
    type_colors = {
        'New TE Embedding': '#2196F3',
        'Prod TE Embedding': '#FF5722',
        'Baseline': '#4CAF50'
    }
    bar_colors = [type_colors.get(t, '#999999') for t in top_20['type']]
    
    ax = axes[ax_idx]
    ax.barh(range(len(top_20)), top_20['importance'].values, color=bar_colors)
    ax.set_yticks(range(len(top_20)))
    ax.set_yticklabels(top_20['feature'].values, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Gain')
    ax.set_title(f'Top 20 Features — {model_name}', fontsize=10)
    ax.grid(True, alpha=0.3, axis='x')
    
    # Print importance breakdown
    imp_df['type'] = imp_df['feature'].apply(
        lambda x: 'New TE Embedding' if x.startswith('embedding_') 
        else ('Prod TE Embedding' if x.startswith('emb') else 'Baseline')
    )
    type_summary = imp_df.groupby('type').agg(
        total_importance=('importance', 'sum'),
        count=('feature', 'count'),
        mean_importance=('importance', 'mean')
    )
    type_summary['pct_importance'] = type_summary['total_importance'] / type_summary['total_importance'].sum() * 100
    print(f"\n{model_name} — Feature Type Breakdown:")
    print(type_summary.to_string())

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2196F3', label='New TE Embedding'),
    Patch(facecolor='#FF5722', label='Prod TE Embedding'),
    Patch(facecolor='#4CAF50', label='Baseline')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=10)

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('models/feature_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Model Artifacts

In [ ]:
# --- Save all model artifacts ---
os.makedirs('models', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

artifact_manifest = []

for model_key, result in [('model1_new_te_only', results_model1),
                           ('model2_new_te_hybrid', results_model2),
                           ('model3_production', results_model3)]:
    prefix = f'medicare_ip_full_eval_{model_key}_{timestamp}'
    
    # Save XGBoost model
    model_path = f'models/{prefix}.model'
    result['model'].save_model(model_path)
    
    # Save parameters
    params_path = f'models/{prefix}_params.pkl'
    with open(params_path, 'wb') as f:
        pickle.dump(result['params'], f)
    
    # Save feature list
    features_path = f'models/{prefix}_features.pkl'
    with open(features_path, 'wb') as f:
        pickle.dump(result['features'], f)
    
    # Save feature importance
    importance = result['model'].get_score(importance_type='gain')
    imp_df = pd.DataFrame([{'feature': k, 'importance': v} for k, v in importance.items()])
    imp_df = imp_df.sort_values('importance', ascending=False)
    imp_path = f'models/{prefix}_importance.csv'
    imp_df.to_csv(imp_path, index=False)
    
    # Save metrics
    metrics_path = f'models/{prefix}_metrics.csv'
    pd.DataFrame([result['metrics']]).to_csv(metrics_path, index=False)
    
    metrics_pkl_path = f'models/{prefix}_metrics.pkl'
    with open(metrics_pkl_path, 'wb') as f:
        pickle.dump(result['metrics'], f)
    
    # Save scaler if used
    if result['scaler'] is not None:
        scaler_path = f'models/{prefix}_scaler.pkl'
        with open(scaler_path, 'wb') as f:
            pickle.dump(result['scaler'], f)
    
    artifact_manifest.append({
        'model_key': model_key,
        'model_path': model_path,
        'n_features': len(result['features']),
        'val_auc_roc': result['metrics']['val_auc_roc'],
        'test_auc_roc': result['metrics']['test_auc_roc'],
        'oot_auc_roc': result['metrics']['oot_auc_roc']
    })
    print(f"Saved artifacts for {model_key}: {prefix}")

# Save comparison summary
df_comparison.to_csv(f'models/full_eval_comparison_intime_{timestamp}.csv', index=False)
df_oot_comparison.to_csv(f'models/full_eval_comparison_oot_{timestamp}.csv', index=False)
df_combined.to_csv(f'models/full_eval_comparison_combined_{timestamp}.csv', index=False)

print(f"\nAll artifacts saved with timestamp: {timestamp}")
print("\nArtifact manifest:")
display(pd.DataFrame(artifact_manifest))

## Summary

### Key Comparisons
| Comparison | What It Tests |
|---|---|
| Model 1 vs Model 2 | Does adding baseline features improve new TE embeddings? |
| Model 2 vs Model 3 | Are new TE embeddings better than production TE embeddings (same baseline)? |
| Model 1 vs Model 3 | How do new TE embeddings alone compare to the full production pipeline? |

### Notes
- All models trained on the **same population** (inner join of both tables) for fair comparison
- Hyperparameters match the original pipeline exactly
- Model 1 uses embeddings-only params (slightly relaxed regularization + `scale_pos_weight`)
- Models 2 & 3 use production params (heavy regularization, no class weighting)
- OOT evaluation tests temporal stability (2025-07-01 to 2025-09-30)